## Experiment 3: Accuracy-Fairness trade-offs
In this notebook, we study whether the $\texttt{badr}$ method has a higher cost on accuracy than other baselines.

## Preliminaries

In [ ]:
# Imports
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
import numpy as np
import jax.numpy as jnp
import pandas as pd
from tqdm.auto import tqdm
import badr

In [ ]:
# Helpers
def accuracy_and_fairness(model, metric, dset, weights):
    model.set_group_weights(weights)
    model.fit(dset.X_train, dset.y_train, dset.groups)
    train_acc = float(model.score(dset.X_train, dset.y_train))
    test_acc = float(model.score(dset.X_test, dset.y_test))
    train_metric = float(metric.fun(model.coef_, dset, train_test="train").item())
    test_metric = float(metric.fun(model.coef_, dset, train_test="test").item())
    return {
        "train_acc": train_acc,
        "test_acc": test_acc,
        "train_metric": train_metric,
        "test_metric": test_metric,
    }


def erm_vals(model, metric, dset):
    weights = jnp.full((len(dset.X_train_list),), 1.0 / len(dset.X_train_list))
    dct = accuracy_and_fairness(model, metric, dset, weights)
    dct["weights"] = np.array(weights, dtype=float)
    return dct


def one_fit_vals(model, metric, dset):
    n_groups = len(dset.X_train_list)
    # basis vectors e_i
    basis = jnp.eye(n_groups, dtype=jnp.float64)

    # find best basis vector (min train_metric)
    best_idx = 0
    best_val = jnp.inf
    for i in range(n_groups):
        wi = basis[i]
        val = accuracy_and_fairness(model, metric, dset, wi)["train_metric"]
        if val < best_val:
            best_val = val
            best_idx = i

    weights = basis[best_idx]
    dct = accuracy_and_fairness(model, metric, dset, weights)
    dct["weights"] = np.array(weights, dtype=float)
    return dct


def balanced_vals(model, metric, dset):
    # Balanced generalized to K >= 2 groups:
    # weight_i = (n - size_i) / ((K-1) * n)  so weights sum to 1 and for K=2 reduces to original formula.
    sizes = [X.shape[0] for X in dset.X_train_list]
    K = len(sizes)
    if K < 2:
        raise ValueError("balanced_vals requires at least 2 groups")
    n = sum(sizes)
    weights = jnp.array([(n - s) / ((K - 1) * n) for s in sizes], dtype=jnp.float64)
    dct = accuracy_and_fairness(model, metric, dset, weights)
    dct["weights"] = np.array(weights, dtype=float)
    return dct


def badr_vals(model, metric, dset, n_points=101):
    # 2D grid over (lmbd1, lmbd2) with third weight = 1 - lmbd1 - lmbd2
    K = len(dset.X_train_list)
    if K != 3:
        raise ValueError("badr_vals 2D-grid currently supports exactly 3 groups (K=3)")

    grid = jnp.linspace(0.0, 1.0, num=n_points)
    best_val = jnp.inf
    best_pair = None

    for l1 in grid:
        for l2 in grid:
            # ensure third weight non-negative
            if float(l1 + l2) > 1.0:
                continue
            w = jnp.array([l1, l2, 1.0 - l1 - l2], dtype=jnp.float64)
            val = accuracy_and_fairness(model, metric, dset, w)["train_metric"]
            if val < best_val:
                best_val = val
                best_pair = (float(l1), float(l2))

    if best_pair is None:
        raise ValueError(
            "No valid (lmbd1, lmbd2) found; try increasing n_points or relax constraints"
        )

    l1, l2 = best_pair
    weights = jnp.array([l1, l2, 1.0 - l1 - l2], dtype=jnp.float64)
    dct = accuracy_and_fairness(model, metric, dset, weights)
    dct["weights"] = np.array(weights, dtype=float)
    return dct


def minmax_vals(model, metric, dset):
    if isinstance(model, badr.models.LogisticRegression):
        mm_model = badr.models.NonsmoothMinMaxLogisticRegression(l2_reg=model.l2_reg)
        mm_model.fit(dset.X_train_list, dset.y_train_list)
    elif isinstance(model, badr.models.SVM):
        mm_model = badr.models.NSMMSVM(l2_reg=model.l2_reg)
        mm_model.fit(dset.X_train, dset.y_train, dset.groups)
    elif isinstance(model, badr.models.RidgeRegression):
        mm_model = badr.models.NSMMRR(l2_reg=model.l2_reg)
        mm_model.fit(dset.X_train_list, dset.y_train_list)
    else:
        raise ValueError("Unsupported model type for minmax_vals")
    gw = jnp.asarray(mm_model.group_weights_, dtype=jnp.float64)
    dct = accuracy_and_fairness(model, metric, dset, gw)
    dct["weights"] = np.array(gw, dtype=float)
    return dct

## Instance configuration

In [ ]:
dsets = {
    "North Dakota": badr.datasets.fetch_ACSTravelTime_cv(
        states="FL", year=2014, n_groups=2, n_splits=2
    ),  # rename as you wish
    "Vermont": badr.datasets.fetch_ACSTravelTime_cv(
        states="NY", year=2014, n_groups=2, n_splits=2
    ),  # placeholder
    "Idaho": badr.datasets.fetch_ACSIncome_cv(
        states="PA", year=2014, n_groups=2, n_splits=2
    ),  # placeholder
    # add/remove datasets here, names are what will appear in the table
}
model = badr.models.LogisticRegression(l2_reg=1e-1)
metric = badr.metrics.IndividualFairness()

methods = ["ERM", "One-Fit", "Balanced", "BADR", "MinMax"]
method_label = {
    "ERM": "ERM",
    "One-Fit": "One-Fit",
    "Balanced": "Balanced",
    "BADR": "BADR",
    "MinMax": "MinMax",
}

# results[method][dataset]["ACC"/"DP"] = (mean, std)
results = {m: {} for m in methods}

In [ ]:
def mean_results(results_list):
    df = pd.DataFrame(results_list)
    return {
        "train_acc": df["train_acc"].mean(),
        "test_acc": df["test_acc"].mean(),
        "train_metric": df["train_metric"].mean(),
        "test_metric": df["test_metric"].mean(),
    }


def std_results(results_list):
    df = pd.DataFrame(results_list)
    return {
        "train_acc": df["train_acc"].std(),
        "test_acc": df["test_acc"].std(),
        "train_metric": df["train_metric"].std(),
        "test_metric": df["test_metric"].std(),
    }


for dset_name, dset_list in dsets.items():
    # normalize to iterable splits (CV loaders already return lists, single datasets do not)
    if not isinstance(dset_list, (list, tuple)):
        dset_iterable = [dset_list]
    else:
        dset_iterable = dset_list

    erm_results = []
    one_fit_results = []
    balanced_results = []
    badr_results = []
    minmax_results = []

    for dset in tqdm(dset_iterable, desc=f"Dataset: {dset_name}"):
        erm_results.append(erm_vals(model, metric, dset))
        one_fit_results.append(one_fit_vals(model, metric, dset))
        balanced_results.append(balanced_vals(model, metric, dset))
        badr_results.append(badr_vals(model, metric, dset))
        minmax_results.append(minmax_vals(model, metric, dset))

    erm_mean, erm_std = mean_results(erm_results), std_results(erm_results)
    one_fit_mean, one_fit_std = (
        mean_results(one_fit_results),
        std_results(one_fit_results),
    )
    balanced_mean, balanced_std = (
        mean_results(balanced_results),
        std_results(balanced_results),
    )
    badr_mean, badr_std = mean_results(badr_results), std_results(badr_results)
    minmax_mean, minmax_std = mean_results(minmax_results), std_results(minmax_results)

    def store(method_name, mean, std):
        results[method_name][dset_name] = {
            "ACC": (mean["test_acc"], std["test_acc"]),
            "DP": (
                mean["test_metric"],
                std["test_metric"],
            ),  # rename to DEO if you prefer
        }

    store("ERM", erm_mean, erm_std)
    store("One-Fit", one_fit_mean, one_fit_std)
    store("Balanced", balanced_mean, balanced_std)
    store("BADR", badr_mean, badr_std)
    store("MinMax", minmax_mean, minmax_std)

# --------- decide which entries to embolden (best per column) ---------

datasets_order = list(dsets.keys())
is_best = {}  # key = (method, dataset, metric) -> bool

for ds in datasets_order:
    # ACC: higher is better
    acc_vals = {m: results[m][ds]["ACC"][0] for m in methods}
    valid_acc = [v for v in acc_vals.values() if not np.isnan(v)]
    best_acc = max(valid_acc) if valid_acc else np.nan

    # DP: lower is better
    eo_vals = {m: results[m][ds]["DP"][0] for m in methods}
    valid_eo = [v for v in eo_vals.values() if not np.isnan(v)]
    best_eo = min(valid_eo) if valid_eo else np.nan

    for m in methods:
        v_acc = acc_vals[m]
        v_eo = eo_vals[m]
        is_best[(m, ds, "ACC")] = not np.isnan(v_acc) and v_acc == best_acc
        is_best[(m, ds, "DP")] = not np.isnan(v_eo) and v_eo == best_eo


def format_cell(method_name, ds_name, metric_name):
    val, std = results[method_name][ds_name][metric_name]
    if np.isnan(val):
        return ""
    best_here = is_best.get((method_name, ds_name, metric_name), False)

    if np.isnan(std):
        base = f"{val:.2f}"
    else:
        base = f"{val:.2f} \\pm {std:.2f}"

    if best_here:
        return f"$\\mathbf{{{base}}}$"
    else:
        return f"${base}$"


# --------- print LaTeX table matching your style ---------

col_spec = "c" + "|cc" * len(datasets_order)

print(r"\begin{table*}[t]")
print(r"\vspace{-1em}")
print(r"\centering")
print(
    r"\caption{\textbf{Performance in binary setting.} Accuracy (ACC) and DP on benchmark datasets.}"
)
print(r"\label{tab:binary_results}")
print(r"\setlength{\tabcolsep}{5pt}")
print(r"\begin{adjustbox}{max width=\textwidth}%{")
print(r"\begin{tabular}{" + col_spec + r"}")
print(r"\toprule")

# first header row: dataset names
header1_parts = [" "]
for i, ds in enumerate(datasets_order):
    if i < len(datasets_order) - 1:
        header1_parts.append(r"\multicolumn{2}{c|}{\textbf{" + ds + "}}")
    else:
        header1_parts.append(r"\multicolumn{2}{c}{\textbf{" + ds + "}}")
print(" & ".join(header1_parts) + r" \\")

# second header row: Method / ACC / DP
header2 = r"\textbf{Method}"
for _ in datasets_order:
    header2 += r" & \textbf{ACC} & \textbf{DP}"
print(header2 + r" \\ \midrule")

# data rows
for m in methods:
    row_parts = [method_label[m]]
    for ds in datasets_order:
        row_parts.append(format_cell(m, ds, "ACC"))
        row_parts.append(format_cell(m, ds, "DP"))
    print(" & ".join(row_parts) + r" \\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{adjustbox}")
print(r"\end{table*}")